# Credit Card Fraud Detection Model

The goal is to create a model to detect fraudulent credit card transactions. The model is built based on an kaggle dataset. \
Link to the data: https://www.kaggle.com/datasets/neharoychoudhury/credit-card-fraud-data

In [ ]:
# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('fraud_data.csv')

In [ ]:
print(df.head())

In [ ]:
df.info()

## First Observation
There are no missing values in the dataset

# EDA (Univariate, Bivariate)

## is_fraud

In [ ]:
df.is_fraud.value_counts()

**Observation:**
- There are two values in the is_fraud column that are timestamps -> delete the two rows
- The clumn is object type -> convert to int

In [ ]:
# get index of the two defect rows
df.loc[~(df.is_fraud.isin(['0','1']))]

In [ ]:
# drop the rows
df = df.drop(index=[1781,7780])
# convert the column to int 
df['is_fraud'] = df.is_fraud.astype('int')

**Conclusion:** \
There are  ~14.5% fraudulent transactions in the dataset

## merchant

In [ ]:
# Check fraud rate for different merchants
merchant_fraud_rate = df.groupby('merchant')['is_fraud'].agg(['count', 'sum', 'mean']).sort_values(by='mean', ascending=False)
merchant_fraud_rate[:10]

**Observation:**\
There are merchants with very high fraud rate but this could be due to random variation. \
**Next** \
To confirm signal -> check distribution for merchant counts and plot count against fraud_rate.

In [ ]:
df.merchant.value_counts().describe()

In [ ]:
sns.scatterplot(data=merchant_fraud_rate, x='count', y='mean')

Most merchants occure about twenty times -> signal possibly noisy

**Next:** Test if variation of fraud rates is random or real signal 

In [ ]:
# calculate the variation of the fraud rates across the merchants
real_std = df.groupby('merchant')['is_fraud'].mean().std()
print(f"Real STD: {round(real_std,2)}")


Shuffle the fraud variable to break every signal that may exists between merchant and fraud. Make multiple simulations then compare the mean random std of the samples to the real std.

In [ ]:
simulated_stds = []

for _ in range(1000):
    shuffled = np.random.permutation(df['is_fraud'])

    std = df.assign(shuffled=shuffled).groupby('merchant')['shuffled'].mean().std()

    simulated_stds.append(std)

print(f"Real STD: {round(real_std,2)}")
print(f"Mean random STD: {round(np.mean(simulated_stds),2)}")
print(f"Max random STD: {round(np.max(simulated_stds),2)}")

**Conclusion:** \
Since the real std is is higher than all the simulated stds it is very likely that merchant is a useful feature. However not every merchant will be reliable due to low counts per merchants. \
**Decision:**\
Use feature in the model but use feature encoding with smoothing to balance out high fraud rate variance in small sample merchants.

## category

In [ ]:
df.category.value_counts()

In [ ]:
# get fraud rate per category
category_fraudrate = df.groupby('category')['is_fraud'].agg(['count','sum','mean']).sort_values('mean', ascending=False)
category_fraudrate

**Conclusion:**\
Clear stable feature here. Use for model with One-Hot-Encoding